# Impulse - ML Feature Engineering Demo

The **Impulse Framework** is a Python library that enables
automotive and industrial engineers to process, aggregate,
and analyze petabytes of time-series measurement data on
Databricks, without requiring Apache Spark expertise.

**What this notebook builds:**
A route classification pipeline. Given 50 test drives
recorded from a Seat Leon, the Impulse Framework extracts
per-drive statistics (min, mean, max of Engine RPM,
Coolant Temperature, and Vehicle Speed) as ML features.
A Random Forest classifier then learns to predict which
route was driven based on the sensor data alone.

The model is tracked with **MLflow** and deployed to
**Model Serving** for real-time route classification.

**Requirements:**
- Databricks workspace with **Unity Catalog** access
- **Serverless** compute (or DBR 14+)

---

## End-to-End Workflow

This notebook follows a 10-step pipeline from raw
measurement data to a deployed real-time classifier.
The diagram reappears before each major step, highlighting
your current position in the pipeline.

![Workflow Overview](images/workflow_overview.png)

# 1. Setup

In [0]:
%pip install pydantic>=2.0 scipy scikit-learn -q
dbutils.library.restartPython()

### Configure target location

Fill in **Catalog**, **Schema**, and **Table Prefix**
in the widgets above, then run the next cells.

In [0]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text(
    "table_prefix", "", "Table Prefix"
)
dbutils.widgets.dropdown(
    "drop_created_tables", "false",
    ["true", "false"], "Drop Created Tables",
)

In [0]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import sys, os

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_PREFIX = dbutils.widgets.get("table_prefix")

if not CATALOG or not SCHEMA:
    raise ValueError(
        "Please set Catalog and Schema "
        "widgets above before running."
    )

nb_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().notebookPath().get()
)
DEMOS_DIR = (
    "/Workspace"
    + "/".join(nb_path.split("/")[:-1])
)
REPO_ROOT = (
    "/Workspace"
    + "/".join(nb_path.split("/")[:-2])
)
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

pfx = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}"
print(f"Target: {pfx}_*")

**Phase: Data Preparation**

![Data Preparation](images/workflow_phase_1.png)

### Load demo data into Silver layer

Impulse reads from a **Silver layer**:
- **Container** = one measurement recording
  (e.g., one test drive)
- **Channel** = one sensor signal within a container
  (e.g., Engine RPM), stored as interval-encoded
  `(tstart, tend, value)` samples

Each container is tagged with a `from_city` and
`to_city` indicating the route driven.

In [0]:
import pandas as pd

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{SCHEMA}"
)

csv_dir = os.path.join(DEMOS_DIR, "data", "ml")

SILVER = [
    "container_metrics", "container_tags",
    "channel_metrics", "channel_tags",
    "channels",
]
for t in SILVER:
    path = f"{csv_dir}/{t}.csv"
    if not os.path.exists(path):
        path = f"{path}.gz"
    pdf = pd.read_csv(path)
    spark.createDataFrame(pdf).write.mode(
        "overwrite"
    ).saveAsTable(f"{pfx}_{t}")

print(f"Loaded {len(SILVER)} tables")

# 2. Initialize the Report

The `Report` orchestrator takes a config specifying:
- **`source`** - Silver layer tables
- **`unity_sink`** - Gold layer output location
- **`query_engine.solver`** - `DeltaSolver` for
  parallel per-container execution
- **`query_engine.data_type`** - `RLE` for
  interval-encoded data (tstart/tend columns)
- **`measurement_dimensions`** - container metadata
  carried into results

In [0]:
from databricks.sdk import WorkspaceClient
from impulse_reporting.core.report import Report
from impulse_reporting.core.page import Page
from impulse_reporting.aggregations.stats_aggregator import StatsAggregator
from impulse_reporting.events.container_event import ContainerEvent

ws = WorkspaceClient()

config = {
    "source": {
        "container_metrics_table": f"{pfx}_container_metrics",
        "channel_metrics_table": f"{pfx}_channel_metrics",
        "channels_uri": f"{pfx}_channels",
        "container_tags_table": f"{pfx}_container_tags",
        "channel_tags_table": f"{pfx}_channel_tags",
    },
    "unity_sink": {
        "catalog": CATALOG,
        "schema": SCHEMA,
        "table_prefix": TABLE_PREFIX,
    },
    "query_engine": {
        "solver": "DeltaSolver",
        "data_type": "RLE",
    },
    "measurement_dimensions": [
        "container_id", "vehicle_key",
        "start_ts", "stop_ts",
    ],
}

report = Report(
    name="ml_feature_engineering", spark=spark, config=config,
    workspace_client=ws
)
db = report.get_db()
print("Report initialized")

**Phase: Feature Engineering (Impulse)**

![Feature Engineering](images/workflow_phase_2.png)

# 3. Select Channels

Channels are selected by **metadata tags** using
the QueryBuilder. No column names, no SQL, no joins.
These are **lazy expressions** and no data is read yet.

We select three signals that characterize driving behavior:
- **Engine RPM** - differs across highway vs city routes
- **Engine Coolant Temperature** - varies with drive duration and load
- **Vehicle Speed** - directly reflects the type of road driven

In [0]:
eng_rpm = db.query.channel(
    channel_name="Engine RPM",
    brand="Seat", model="Leon",
)
coolant_temp = db.query.channel(
    channel_name="Engine Coolant Temperature",
    brand="Seat", model="Leon",
)
veh_spd = db.query.channel(
    channel_name="Vehicle Speed Sensor",
    brand="Seat", model="Leon",
)

channels = [eng_rpm, coolant_temp, veh_spd]
channel_names = [
    "Engine RPM", "Coolant Temp", "Vehicle Speed",
]

# 4. Define Event & Aggregation

A **ContainerEvent** spans the full duration of each
measurement recording, so every sample is in scope.

The **StatsAggregator** aggregation computes min, mean, and
max per channel per container. These per-drive summaries
become the feature vector for route classification.

In [ ]:
container_event = ContainerEvent(
    name="container_event",
    desc="Full measurement recording",
)
report.add_event(container_event)

page = Page(page_number=1)
report.add_page(page)

page.add_aggregation(StatsAggregator(
    name="drive_stats",
    input_expressions=channels,
    channel_names=channel_names,
    statistics=["min", "mean", "max"],
    event=container_event,
    desc="Per-drive statistics for route classification",
))
print("Event and aggregation registered")

# 5. Compute Statistics

`determine_report()` runs the statistics computation
in parallel across all containers via Spark Pandas UDFs.
No Gold-layer write is needed here since we retrieve
the results directly from memory.

In [0]:
report.determine_report()
print("Statistics computed")

**Phase: ML Pipeline**

![ML Pipeline](images/workflow_phase_3.png)

# 6. Extract Feature Matrix

The raw statistics DataFrame has one row per
(container, channel, statistic). We display it first,
then pivot it into a feature vector suitable for ML.

In [0]:
stats_bundle = report.aggregation_dfs["STATS_AGGREGATOR"]
stats_df = stats_bundle.get("changed") or stats_bundle.get("unchanged")
display(stats_df)


### Build route labels

Each container is tagged with `from_city` and `to_city`.
We combine them into a route label (e.g. `RT->KA`).
Routes with fewer than 3 recordings are grouped as `OTHER`.

In [0]:
import pyspark.sql.functions as F

features_df = (
    stats_df
    .withColumn(
        "feature",
        F.concat(
            F.col("channel_name"), F.lit("_"),
            F.col("aggregation_label"),
        ),
    )
    .groupBy("container_id")
    .pivot("feature")
    .agg(F.first("statistic_value"))
)

tags_df = spark.read.table(f"{pfx}_container_tags")
from_city = tags_df.filter(F.col("key") == "from_city").select(
    F.col("container_id"), F.col("value").alias("from_city")
)
to_city = tags_df.filter(F.col("key") == "to_city").select(
    F.col("container_id"), F.col("value").alias("to_city")
)
routes_df = from_city.join(to_city, "container_id").withColumn(
    "route", F.concat(F.col("from_city"), F.lit("->"), F.col("to_city"))
)

MIN_SAMPLES = 3
route_counts = routes_df.groupBy("route").count()
common_routes = [
    r["route"] for r in route_counts.filter(F.col("count") >= MIN_SAMPLES).collect()
]
routes_df = routes_df.withColumn(
    "route",
    F.when(F.col("route").isin(common_routes), F.col("route")).otherwise("OTHER"),
)

labeled_df = features_df.join(
    routes_df.select("container_id", "route"), "container_id"
)
display(labeled_df.groupBy("route").count().orderBy(F.desc("count")))

# 7. Train Route Classifier

We split the data into **80% train / 20% test**
(stratified by route) and train a scikit-learn
**Random Forest** classifier, tracked with **MLflow**.

Cross-validation is performed on the training set only.
The held-out test set is used for final evaluation.

In [0]:
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score, train_test_split
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")
mlflow.sklearn.autolog(log_datasets=False)

feature_cols = sorted([
    c for c in labeled_df.columns
    if c not in ("container_id", "route")
])
all_pdf = labeled_df.select(
    "container_id", "route", *feature_cols
).toPandas().dropna(subset=feature_cols)

le = LabelEncoder()
all_pdf["route_encoded"] = le.fit_transform(all_pdf["route"])

train_pdf, test_pdf = train_test_split(
    all_pdf, test_size=0.2,
    stratify=all_pdf["route_encoded"],
    random_state=42,
)

X_train = train_pdf[feature_cols]
y_train = train_pdf["route_encoded"]
X_test = test_pdf[feature_cols]
y_test = test_pdf["route_encoded"]

with mlflow.start_run(run_name="route_classifier") as run:
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
    )
    model.fit(X_train, y_train)

    cv_scores = cross_val_score(model, X_train, y_train, cv=3)
    mlflow.log_metric("cv_accuracy_mean", float(cv_scores.mean()))
    mlflow.log_metric("cv_accuracy_std", float(cv_scores.std()))
    mlflow.log_metric("n_routes", len(le.classes_))
    mlflow.log_metric("n_train", len(X_train))
    mlflow.log_metric("n_test", len(X_test))

    signature = infer_signature(X_train, model.predict(X_train))
    mlflow.sklearn.log_model(
        model, "route_classifier",
        signature=signature,
    )

    run_id = run.info.run_id

print(f"MLflow run: {run_id}")
print(f"Routes: {list(le.classes_)}")
print(f"Train: {len(X_train)} drives, Test: {len(X_test)} drives")
print(f"CV accuracy (train): {cv_scores.mean():.2f} (+/- {cv_scores.std():.2f})")

# 8. Evaluate Results

Predicted vs actual route on the **held-out test set**.
These drives were not seen during training.

In [0]:
test_pdf["predicted_route"] = le.inverse_transform(
    model.predict(X_test)
)
test_pdf["correct"] = (
    test_pdf["route"] == test_pdf["predicted_route"]
)

result_df = spark.createDataFrame(
    test_pdf[["container_id", "route", "predicted_route", "correct"]]
).orderBy("container_id")
display(result_df)

accuracy = test_pdf["correct"].mean()
print(f"Test accuracy: {accuracy:.0%} ({test_pdf['correct'].sum()}/{len(test_pdf)})")

**Phase: Deployment**

![Deployment](images/workflow_phase_4.png)

# 9. Register & Deploy to Model Serving

Register the trained model in **Unity Catalog** using
a **serverless optimized deployment**. The `env_pack`
parameter pre-packages model artifacts and the notebook
environment during registration, so endpoint creation
skips the container build and deploys much faster.

> **Note:** Creating a new endpoint can take ~5 minutes
> while the serving infrastructure is provisioned.

In [0]:
from mlflow.utils.env_pack import EnvPackConfig

model_name = f"{CATALOG}.{SCHEMA}.mda_route_classifier"

model_uri = f"runs:/{run_id}/route_classifier"
mv = mlflow.register_model(
    model_uri, model_name,
    env_pack=EnvPackConfig(name="databricks_model_serving"),
)
print(f"Registered: {model_name} v{mv.version}")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

w = WorkspaceClient()
endpoint_name = f"{TABLE_PREFIX}_route_classifier"

served_entity = ServedEntityInput(
    entity_name=model_name,
    entity_version=mv.version,
    scale_to_zero_enabled=True,
    workload_size="Small",
)

try:
    w.serving_endpoints.create_and_wait(
        name=endpoint_name,
        config=EndpointCoreConfigInput(
            served_entities=[served_entity],
        ),
    )
except Exception as e:
    if "already exists" in str(e):
        print(f"Endpoint exists, updating model to v{mv.version}...")
        w.serving_endpoints.update_config_and_wait(
            name=endpoint_name,
            served_entities=[served_entity],
        )
    else:
        raise

host = spark.conf.get("spark.databricks.workspaceUrl")
url = f"https://{host}/ml/endpoints/{endpoint_name}"
print(f"Endpoint: {url}")
displayHTML(
    f'<a href="{url}" target="_blank">'
    f"Open Serving Endpoint</a>"
)

### Test the endpoint

Score the held-out test drives against the deployed
model via REST. The model returns the encoded route
label, which we map back to the route name.

In [0]:
from databricks.sdk.service.serving import DataframeSplitInput

for _, row in test_pdf.iterrows():
    features = row[feature_cols].to_frame().T
    split = features.to_dict(orient="split")

    response = w.serving_endpoints.query(
        name=endpoint_name,
        dataframe_split=DataframeSplitInput(
            columns=split["columns"],
            data=split["data"],
        ),
    )

    predicted = le.inverse_transform(
        [int(response.predictions[0])]
    )[0]
    actual = row["route"]
    match = "OK" if predicted == actual else "MISS"
    print(
        f"Container {row['container_id']}: "
        f"actual={actual}, predicted={predicted} [{match}]"
    )

# 10. Cleanup

Drop all tables created by this demo.

In [0]:
if dbutils.widgets.get("drop_created_tables") == "true":
    for t in SILVER:
        spark.sql(f"DROP TABLE IF EXISTS {pfx}_{t}")
        spark.sql(f"DROP TABLE IF EXISTS {pfx}_stats_fact")
        spark.sql(f"DROP TABLE IF EXISTS {pfx}_stats_dimension")
        spark.sql(f"DROP TABLE IF EXISTS {pfx}_event_instance_fact")
        spark.sql(f"DROP TABLE IF EXISTS {pfx}_event_dimension")
        spark.sql(f"DROP TABLE IF EXISTS {pfx}_measurement_dimension")
        print("Tables dropped")